# Delivery Performance Analysis

## Ziel

In diesem Notebook wird die Lieferperformance anhand der vorliegenden Logistikdaten untersucht. Ziel ist es, Lieferzeiten und Verspätungen zu analysieren und mögliche Unterschiede zwischen Regionen, Transportarten und Produkten zu identifizieren.

Dabei werden folgende Fragestellungen untersucht:

1. Wie lange dauern Lieferungen durchschnittlich?
2. Welche Regionen haben besonders lange Lieferzeiten?
3. Welche Transportarten sind besonders schnell bzw. langsam?
4. Wie häufig kommen Lieferungen verspätet an?
5. Gibt es bestimmte Regionen oder Produkte mit überdurchschnittlich vielen Verspätungen?

Zur Beantwortung der Fragen werden die relevanten Tabellen zunächst zusammengeführt und anschließend bereinigt und analysiert. Die Ergebnisse werden durch geeignete Visualisierungen dargestellt.

## Datenbasis

Für die Analyse werden insbesondere folgende Tabellen verwendet:

- `loads` – Informationen zu den Sendungen
- `trips` – Informationen zu den durchgeführten Transporten
- `delivery_events` – Informationen zu Abholung und Lieferung
- `routes` – Informationen zu den jeweiligen Routen

Die Tabellen stehen in folgender Beziehung zueinander:

- `delivery_events`-(`trip_id`)->`trips`-(`load_id`)->`loads`-(`route_id`)->`routes`
- `delivery_events`-(`load_id`)->`loads`


## 1. Daten laden

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data/raw")

loads = pd.read_csv(DATA_DIR / "loads.csv")
trips = pd.read_csv(DATA_DIR / "trips.csv")
delivery_events = pd.read_csv(DATA_DIR / "delivery_events.csv")
routes = pd.read_csv(DATA_DIR / "routes.csv")


## 2. Tabellen zusammenführen

In [2]:
# Merge zusammengefasst
final_df = (
    loads
    .merge(routes, how="left", on="route_id")
    .merge(trips, how="left", on="load_id")
    .merge(delivery_events, how="left", on="trip_id")
)
# get size of loads and final (rows, columns) and print them to console
# goal is to check if there are more rows or columns after merging tables
print("Loads:", loads.shape)
print("Final:", final_df.shape)

# count frequency of event-types, accesses column "event_type" of the dataframe
# goal: get number of pickup-events and delivery-events
print("\nEvent types:")
print(delivery_events["event_type"].value_counts())

# groupby -> divide data in on column in distinct groups
# describe -> statistical summary
# goal: get the number of events per load
print("\nEvents per load:")
print(delivery_events.groupby("load_id").size().describe())

# isna -> check each cell if there is an empty or NaN-value -> converts it to True (missing) and False (not missing)
# sum -> counts these values (True equals 1, False equals 0)
# sort_values -> sort values according to number of missing values
# head(n) -> show the first n values
print("\nMissing values:")
print(final_df.isna().sum().sort_values(ascending=False).head(20))


Loads: (85410, 12)
Final: (170820, 41)

Event types:
event_type
Pickup      85410
Delivery    85410
Name: count, dtype: int64

Events per load:
count    85410.0
mean         2.0
std          0.0
min          2.0
25%          2.0
50%          2.0
75%          2.0
max          2.0
dtype: float64

Missing values:
driver_id                 3428
trailer_id                3360
truck_id                  3344
customer_id                  0
load_type                    0
weight_lbs                   0
route_id                     0
load_date                    0
load_id_x                    0
fuel_surcharge               0
revenue                      0
pieces                       0
accessorial_charges          0
origin_state                 0
load_status                  0
booking_type                 0
origin_city                  0
typical_distance_miles       0
destination_state            0
destination_city             0
dtype: int64


## 4. Create new table

Goal in this part is to simplify and clean up the table a bit, so that for each load_id there is a column `pickup_actual_datetime` and a column `delivery_actual_datetime` and each load_id appears only once in the table

In [3]:
analysis_df = final_df.copy()

## 5. Create new columns and add data accordingly

Goal in this part is to simplify and clean up the table a bit, so that for each load_id there is a column `pickup_actual_datetime`, a column `pickup_scheduled_datetime`, a column `delivery_actual_datetime` and a column `delivery_scheduled_datetime` and each load_id appears only once in the table

In [4]:
analysis_df['scheduled_datetime'] = pd.to_datetime(analysis_df['scheduled_datetime'])
analysis_df['actual_datetime'] = pd.to_datetime(analysis_df['actual_datetime'])


## 4. Events auf Load-Ebene transformieren

Die Tabelle `delivery_events` enthält pro Load jeweils ein Pickup- und
ein Delivery-Event. Für die weitere Analyse wird daher eine flache
Struktur mit einer Zeile pro `load_id` erzeugt.

Event-spezifische Informationen wie Zeitpunkte, Standzeiten und
Standorte werden mittels einer Pivot-Tabelle in separate Pickup- und
Delivery-Spalten überführt.

In [5]:
analysis_df.rename(columns={"load_id_x": "load_id"}, inplace=True)
if "load_id_y" in analysis_df.columns:
    analysis_df.drop(columns=["load_id_y"], inplace=True)

events_pivoted = analysis_df.pivot_table(
    index="load_id",
    columns="event_type",
    values=[
        "scheduled_datetime", 
        "actual_datetime", 
        "detention_minutes", 
        "on_time_flag", 
        "facility_id",
        "location_city",
        "location_state"
    ],
    aggfunc="first"
)

events_pivoted.columns = [f"{col[1].lower()}_{col[0]}" for col in events_pivoted.columns]
events_pivoted = events_pivoted.reset_index()

event_cols = [
    "event_id", "event_type", "facility_id", "scheduled_datetime", 
    "actual_datetime", "detention_minutes", "on_time_flag", 
    "location_city", "location_state"
]

load_base_info = analysis_df.drop(columns=event_cols).groupby("load_id", as_index=False).first()

final_flat_df = pd.merge(
    load_base_info, 
    events_pivoted, 
    on="load_id", 
    how="left"
)

print("Zeilenursprung (mit doppelten Events):", len(analysis_df))
print("Neuer Status (exakt 1 Zeile pro Ladung):", len(final_flat_df))
print("Anzahl Spalten gesamt:", len(final_flat_df.columns))

display(final_flat_df.head())
display(final_flat_df.info())

Zeilenursprung (mit doppelten Events): 170820
Neuer Status (exakt 1 Zeile pro Ladung): 85410
Anzahl Spalten gesamt: 45


,load_id,customer_id,route_id,load_date,load_type,weight_lbs,pieces,revenue,fuel_surcharge,accessorial_charges,...,delivery_facility_id,pickup_facility_id,delivery_location_city,pickup_location_city,delivery_location_state,pickup_location_state,delivery_on_time_flag,pickup_on_time_flag,delivery_scheduled_datetime,pickup_scheduled_datetime
0,LOAD00000001,CUST00183,RTE00019,2022-01-01,Dry Van,19178,13,3045.23,406.72,100,...,FAC00046,FAC00034,Detroit,Houston,MI,TX,True,False,2022-01-02 23:10:55.918185,2022-01-01 18:00:00
1,LOAD00000002,CUST00076,RTE00058,2022-01-01,Dry Van,27761,22,1224.48,98.61,0,...,FAC00050,FAC00015,Indianapolis,Kansas City,IN,MO,False,True,2022-01-02 02:13:26.608430,2022-01-01 18:00:00
2,LOAD00000003,CUST00027,RTE00048,2022-01-01,Refrigerated,35594,16,7171.12,792.88,0,...,FAC00022,FAC00001,Portland,Columbus,OR,OH,True,True,2022-01-03 04:28:02.169634,2022-01-01 08:00:00
3,LOAD00000004,CUST00088,RTE00013,2022-01-01,Refrigerated,33274,10,1308.20,141.33,50,...,FAC00037,FAC00029,Denver,Phoenix,CO,AZ,False,False,2022-01-02 06:38:56.020030,2022-01-01 17:00:00
4,LOAD00000005,CUST00185,RTE00020,2022-01-01,Dry Van,40257,10,3317.18,738.48,0,...,FAC00050,FAC00046,Seattle,Houston,WA,TX,True,False,2022-01-02 22:51:35.544202,2022-01-01 08:00:00


<class 'pandas.DataFrame'>
RangeIndex: 85410 entries, 0 to 85409
Data columns (total 45 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   load_id                      85410 non-null  str           
 1   customer_id                  85410 non-null  str           
 2   route_id                     85410 non-null  str           
 3   load_date                    85410 non-null  str           
 4   load_type                    85410 non-null  str           
 5   weight_lbs                   85410 non-null  int64         
 6   pieces                       85410 non-null  int64         
 7   revenue                      85410 non-null  float64       
 8   fuel_surcharge               85410 non-null  float64       
 9   accessorial_charges          85410 non-null  int64         
 10  load_status                  85410 non-null  str           
 11  booking_type                 85410 non-null  str    

None

In [6]:
print("Originale Event-Zeilen:", len(analysis_df))
print("Loads nach Transformation:", len(final_flat_df))

print("\nAnzahl Zeilen pro load_id:")
print(final_flat_df["load_id"].value_counts().value_counts())

print("\nPickup / Delivery pro Load:")
counts = pd.crosstab(
    delivery_events["load_id"],
    delivery_events["event_type"]
)

print(
    ((counts["Pickup"] == 1) & (counts["Delivery"] == 1)).value_counts()
)

print("\nFehlende Event-Zeitpunkte:")
print(
    final_flat_df[
        [
            "pickup_actual_datetime",
            "pickup_scheduled_datetime",
            "delivery_actual_datetime",
            "delivery_scheduled_datetime"
        ]
    ].isna().sum()
)

Originale Event-Zeilen: 170820
Loads nach Transformation: 85410

Anzahl Zeilen pro load_id:
count
1    85410
Name: count, dtype: int64

Pickup / Delivery pro Load:
True    85410
Name: count, dtype: int64

Fehlende Event-Zeitpunkte:
pickup_actual_datetime         0
pickup_scheduled_datetime      0
delivery_actual_datetime       0
delivery_scheduled_datetime    0
dtype: int64


## 5. Lieferperformance – Berechnung und Validierung

In [7]:
calculation_df = final_flat_df.copy()

## 1. Lieferdauer berechnen
Benötigte Spalten `pickup_actual_datetime` und `delivery_actual_datetime`
* `pickup_actual_datetime` < `delivery_actual_datetime` für alle `load_id`
* Erstellung neuer Spalte `delivery_duration_hours`
* Berechnung `delivery_duration_hours` = `delivery_actual_datetime` - `pickup_actual_datetime`
* Vergleich der berechneten Daten mit der bereits vorhandenen Spalte `actual_duration_hours` 

In [8]:
calculation_df['delivery_duration_hours'] = (
    (calculation_df['delivery_actual_datetime'] - calculation_df['pickup_actual_datetime']
).dt.total_seconds() / 3600
).round(2)

In [9]:
target_loads = calculation_df.loc[
    calculation_df['delivery_actual_datetime'] <= calculation_df['pickup_actual_datetime'],
    'load_id'
].tolist()


In [ ]:
invalid_trips = calculation_df[
    calculation_df["delivery_actual_datetime"]
    < calculation_df["pickup_actual_datetime"]
].copy()

error_rate = len(invalid_trips) / len(calculation_df) * 100

print(
    f"Zeitlich inkonsistente Loads: "
    f"{len(invalid_trips)} ({error_rate:.2f}%)"
)


display(
    invalid_trips[
        [
            "load_id",
            "pickup_actual_datetime",
            "delivery_actual_datetime",
            "delivery_duration_hours"
        ]
    ].head()
)

### Datenbereinigung

Bei 486 von 85.410 Loads (0,57 %) liegt der tatsächliche
Delivery-Zeitpunkt vor dem tatsächlichen Pickup-Zeitpunkt.

Da diese zeitliche Reihenfolge nicht plausibel ist, werden die
betroffenen Loads aus den folgenden Analysen ausgeschlossen.

Die Rohdaten bleiben unverändert.

In [ ]:
clean_df = calculation_df[
    calculation_df["delivery_actual_datetime"]
    >= calculation_df["pickup_actual_datetime"]
].copy()

print(f"Loads vor Bereinigung: {len(calculation_df)}")
print(f"Loads nach Bereinigung: {len(clean_df)}")

In [ ]:
diff = (
    clean_df["delivery_duration_hours"]
    - clean_df["actual_duration_hours"]
).abs()

print(f"Durchschnittliche Abweichung: {diff.mean():.2f} Stunden")
print(f"Maximale Abweichung: {diff.max():.2f} Stunden")

Die im Datensatz vorhandene `actual_duration_hours` weicht teilweise deutlich von der aus den Event-Zeitpunkten berechneten Dauer ab. Für die weitere Analyse wird daher die aus Pickup- und Delivery-Zeitpunkt berechnete Lieferdauer verwendet.

Rundungseffekt: Eine Lieferung weist eine zeitliche Inkonsistenz von wenigen Sekunden auf. Da delivery_duration_hours auf Stunden gerundet wurde, ergibt sich hier 0.0 statt eines negativen Werts. Dieser Datensatz wird ebenfalls als inkonsistent behandelt.

In [ ]:
# Dieser Befehl isoliert exakt die eine Zeile, die durchs Netz gegangen ist:
special_case = calculation_df.loc[
    (calculation_df['delivery_actual_datetime'] < calculation_df['pickup_actual_datetime']) & 
    (calculation_df['delivery_duration_hours'] >= 0),
    ['load_id', 'pickup_actual_datetime', 'delivery_actual_datetime', 'delivery_duration_hours']
]

display(special_case)

Die Plausibilitätsprüfung wurde anhand der zugrunde liegenden Zeitstempel durchgeführt und nicht anhand der bereits gerundeten Lieferdauer, da Rundungen zu Grenzfällen wie -0.0 führen können.  
Ein Datensatz ist valide, wenn der tatsächliche Lieferzeitpunkt gleich oder nach dem tatsächlichen Abholzeitpunkt liegt.  


In [ ]:
clean_df = calculation_df[
    calculation_df['delivery_actual_datetime'] >= calculation_df['pickup_actual_datetime']
].copy()

print("Vor Bereinigung:", calculation_df.shape)
print("Nach Bereinigung:", clean_df.shape)


### 1.1 Wie lange dauern Lieferungen durchschnittlich?
### 1.2 Wie haeufig kommen Lieferungen verspaetet an?
### 1.3 Wie groß sind die Verspaetungen?

In [ ]:
delivery_duration = clean_df['delivery_duration_hours'].describe()

is_delayed_delivery = clean_df['delivery_actual_datetime'] > clean_df['delivery_scheduled_datetime']
percentage_delayed_deliveries = (((is_delayed_delivery.values.sum())/(clean_df.shape[0]))*100)

#delivery_delay_duration = clean_df['delivery_actual_datetime'] - clean_df['delivery_scheduled_datetime']
# delayed_trips = clean_df[is_delayed_delivery]

clean_df['delivery_delay'] = (
    (
        clean_df['delivery_actual_datetime'] - clean_df['delivery_scheduled_datetime']
).dt.total_seconds() / 3600
).round(2)


delayed_trips = clean_df[is_delayed_delivery]

#display(delayed_trips.head())

delayed_trips_summary = clean_df.loc[
    is_delayed_delivery, 
    ['load_id', 'delivery_scheduled_datetime', 'delivery_actual_datetime', 'delivery_delay']
]

clean_df['is_delayed_delivery'] = (
    clean_df['delivery_actual_datetime'] > clean_df['delivery_scheduled_datetime']
)
print(clean_df['is_delayed_delivery'].value_counts())

clean_df['delivery_delay'] = (
    clean_df['delivery_actual_datetime']
    - clean_df['delivery_scheduled_datetime']
).dt.total_seconds() / 3600

clean_df['delivery_delay'] = clean_df['delivery_delay'].round(2)

clean_df['is_delayed_delivery'] = clean_df['delivery_delay'] > 0

display(delivery_duration.round(2))
display(is_delayed_delivery.value_counts())
print(f"Anzahl aller Lieferungen: {clean_df.shape[0]}")
print(f"Anzahl verspaeteter Lieferungen: {is_delayed_delivery.values.sum()}")
print(f"Prozentualer Anteil verspaeteter Lieferungen: {percentage_delayed_deliveries:.2f}%")
print("Statistik verspaeteter Lieferungen\n ", clean_df[is_delayed_delivery].delivery_delay.describe().round(2))



## Grafiken

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.hist(
    clean_df['delivery_duration_hours'],
    bins=30
)

plt.xlabel('Lieferdauer (Stunden)')
plt.ylabel('Anzahl Lieferungen')
plt.title('Verteilung der Lieferdauer')

plt.show()



In [ ]:
delay_counts = clean_df['is_delayed_delivery'].value_counts()

plt.figure(figsize=(8, 5))

plt.bar(
    ['Puenktlich / frueh', 'Verspaetet'],
    [
        delay_counts.get(False, 0),
        delay_counts.get(True, 0)
    ]
)

plt.ylabel('Anzahl Lieferungen')
plt.title('Puenktlichkeit der Lieferungen')

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.hist(
    clean_df.loc[clean_df['is_delayed_delivery'], 'delivery_delay'],
    bins=30
)

plt.xlabel('Verspaetung (Stunden)')
plt.ylabel('Anzahl Lieferungen')
plt.title('Verteilung der Verspaetungsdauer')

plt.show()